In [13]:
import json
from pathlib import Path
from datasets import Dataset, Features, Value
from typing import List # Added for the load_txt_lines function

# Dynamically find the monorepo root (where pyproject.toml is)
_p = Path.cwd().resolve()
DATA_DIR = None
while _p != _p.parent:
    if (_p / "pyproject.toml").is_file():
        DATA_DIR = _p / "data"
        break
    _p = _p.parent
if DATA_DIR is None:
    DATA_DIR = Path.cwd().resolve() / "data"


def load_txt_lines(data_dir: Path) -> List[str]:
    lines = []
    for path in sorted(data_dir.glob("*.txt")):
        try:
            raw = path.read_text(encoding="utf-8")
            for line in raw.splitlines():
                s = line.strip()
                if s:
                    lines.append(s)
        except OSError as e:
            print(f"  Warning: could not read {path}: {e}")
    return lines

# Load the raw text data
raw_data = load_txt_lines(DATA_DIR)

print(f"Loaded {len(raw_data):,} lines from .txt files in {DATA_DIR}")

Loaded 663,678 lines from .txt files in /Users/roqqu/Desktop/Build An LLM/build-an-llm/data


In [14]:
import datetime

# Define dataset metadata
# IMPORTANT: Replace these placeholders with your actual information
dataset_name = "9ja-bookcorpus" # e.g., "pdf-scraped-corpus"
huggingface_username = "theKingslee" # Your Hugging Face username

metadata = {
    "name": dataset_name,
    "pretty_name": "160 Books from Nigerian Authors", # A more human-readable name
    "short_description": "A corpus of text data scraped from 160 PDFs from Nigerian Authors, cleaned and refined.",
    "long_description": """
This dataset contains text extracted from a collection of PDF documents.
It has undergone several refinement steps including:
- Basic sanity checks (e.g., mostly alphabetic characters, reasonable word lengths).
- Language detection to filter for English text.
- Spell correction using NLTK's English word corpus.

This dataset is intended for use in language modeling, text generation, and other NLP tasks.
""",
    "license": "apache-2.0", # Choose an appropriate license (e.g., "MIT", "cc-by-4.0")
    "tags": ["text", "ocr", "language-modeling", "english", "pdf-scrape", "nigerian", "9ja-context"],
    "languages": ["en"],
    "citation": "",
    "dataset_info": {
        "features": {
            "text": "The raw text content of each refined sentence/chunk."
        },
        "splits": {
            "train": {
                "name": "train",
                "num_bytes": 0, # To be filled in after dataset creation
                "num_examples": 0, # To be filled in after dataset creation
                "dataset_name": dataset_name
            }
        },
        "version": "1.0.0",
        "date": datetime.datetime.now().strftime("%Y-%m-%d"),
        "processed_by": huggingface_username
    }
}

print("Dataset metadata defined.")

Dataset metadata defined.


In [15]:
from huggingface_hub import DatasetCard, DatasetCardData

# Prepare DatasetCardData from the metadata dictionary
card_data = DatasetCardData(
    license=metadata["license"],
    tags=metadata["tags"],
    language=metadata["languages"][0], # Changed from 'languages' to 'language' and took the first element
    pretty_name=metadata["pretty_name"],
    short_description=metadata["short_description"],
    long_description=metadata["long_description"],
    # Add other fields as needed for your specific dataset
)

# Generate the DatasetCard (README.md content)
card = DatasetCard.from_template(card_data)

# Save the card to a temporary README.md file
# This file will be pushed to the Hub along with the dataset
_readme_path = Path("README.md") # Creates in the current working directory
with _readme_path.open("w", encoding="utf-8") as f:
    f.write(card.content)

print(f"Generated DatasetCard and saved to {_readme_path}")

Generated DatasetCard and saved to README.md


In [16]:
# Create a Hugging Face Dataset
# For a simple text dataset, we define a single 'text' feature.
features = Features({
    'text': Value(dtype='string', id=None)
})

hf_dataset = Dataset.from_dict({"text": raw_data}, features=features)

print(f"Created Hugging Face Dataset with {len(hf_dataset)} examples.")
print("Sample example:", hf_dataset[0])

Created Hugging Face Dataset with 663678 examples.
Sample example: {'text': 'WORLD FANTASY, AWARD-WINNING AUTHOR INNEDI 3% OKORAFOR KATA “There’s more vivid imagination in a-page of Nnedi “Okorafor’s work than in whole volumes of" ordinary fantasy epics.” —UrsuLa K.'}


In [17]:
## Upload to Hugging Face Hub

# IMPORTANT: Replace "your-username/your-dataset-name" with your desired repository ID!
repository_id = f"{huggingface_username}/{dataset_name}" # Using defined metadata

try:
    # Push the dataset
    hf_dataset.push_to_hub(repository_id, commit_message="Initial dataset upload with metadata", 
                           # private=True, # Uncomment to make the dataset private
                           # token="hf_YOUR_TOKEN", # Uncomment if you need to pass a token directly
                          )
    # Upload the DatasetCard (README.md) to the repo
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_file(
        path_or_fileobj=_readme_path, # Path to the generated README.md
        path_in_repo="README.md",
        repo_id=repository_id,
        repo_type="dataset",
        commit_message="Update Dataset Card"
    )

    print(f"Dataset and Dataset Card pushed to https://huggingface.co/datasets/{repository_id}")
except Exception as e:
    print(f"Error pushing to hub: {e}")
    print("Please ensure you are logged into Hugging Face and have write permissions for the repository.")
    print("You can log in by running `huggingface-cli login` in your terminal or `from huggingface_hub import notebook_login; notebook_login()` in a notebook.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  4.24ba/s]
Processing Files (1 / 1): 100%|██████████| 33.1MB / 33.1MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.17s/ shards]
No files have been modified since last commit. Skipping to prevent empty commit.


Dataset and Dataset Card pushed to https://huggingface.co/datasets/theKingslee/9ja-bookcorpus
